# Experiment C — Gemma-2-2B cleanup (strict action parsing)

**Τι ελέγχουμε:** αν τα αρνητικά Δ του Gemma-2-2B είναι συμπεριφορά ή instruction-following breakdown. Με `--action-retries 1`, ένα invalid action ξαναδειγματίζεται μία φορά πριν καταγραφεί ως invalid.

**Pilot πρώτα:** το χειρότερο cell (silence, SH, 45% invalid). Αν τα invalids πέσουν <5%, τρέξε το πλήρες δεύτερο cell. Αν όχι, ο δρόμος είναι η αντικατάσταση με Llama-3.2-3B.

**Κανόνας απόφασης:** invalids <5% ΚΑΙ τα αρνητικά Δ επιμένουν → πραγματική συμπεριφορά. Invalids <5% και τα Δ ομαλοποιούνται → ήταν artifact.

## Setup — install, GPU check, clone, HF token

1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Add-ons → Secrets → `HF_TOKEN` (χρειάζεται για Llama/Gemma)

**Προσοχή:** το repo πρέπει να έχει γίνει push με τις αλλαγές του Phase 1.5 (topologies + `--action-retries`) πριν τρέξει αυτό το notebook.

In [ ]:
!pip install -q bitsandbytes python-dotenv

import torch
assert torch.cuda.is_available(), 'GPU not enabled!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
GITHUB_REPO = 'https://github.com/stsimpe/cheaptalk_bench.git'
import os
if not os.path.exists('/kaggle/working/repo'):
    !git clone $GITHUB_REPO /kaggle/working/repo
%cd /kaggle/working/repo
# sanity: Phase-1.5 features present
assert 'clique' in open('topology.py').read(), 'Repo lacks Phase-1.5 topologies — push first!'
!ls

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HUGGINGFACE_API_KEY'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token loaded')
except Exception as e:
    print('No HF_TOKEN secret (fine for Qwen):', e)

In [ ]:
MODEL_ID = 'google/gemma-2-2b-it'

In [ ]:
# PILOT: το χειρότερο cell μόνο
!python run_all_scenarios.py --provider local --model-id $MODEL_ID \
    --scenarios silence --n-runs 5 --max-new-tokens 256 \
    --action-retries 1 \
    --out-dir-base /kaggle/working/results_gemma2b_strict \
    --zip-after-each --zip-mirror /kaggle/working

In [ ]:
# ΑΝ ο pilot καθαρίσει: όλα τα content-degraded cells
# !python run_all_scenarios.py --provider local --model-id $MODEL_ID \
#     --scenarios silence no_sense counterfactual framing_team \
#     --n-runs 5 --max-new-tokens 256 --action-retries 1 \
#     --out-dir-base /kaggle/working/results_gemma2b_strict \
#     --zip-after-each --zip-mirror /kaggle/working